# Этап B: выдачи всех поисковых конфигураций

Запускается после того, как набор запросов собран целиком: синтетические
с этапа A плюс пятьдесят написанных руками, с проверенной разметкой.

Подключить **три** входа: датасет `ru-fin-retrieval`, датасет
`ru-fin-queries` и вывод ноутбука этапа A (вкладка **Your Work**).
меню **Settings** → **Accelerator** → **GPU T4 ×2**; там же проверить,
что интернет включён (пункт читается как **Turn off internet**).


In [ ]:
!pip install -q -U sentence-transformers


In [ ]:
import glob, os, shutil, zipfile

# Путь к датасету зависит от того, как он назван, а разложить его Kaggle
# может и папкой, и архивом. Поэтому источник ищется по содержимому:
# нужен каталог, где лежит пакет rufin или корпус.
def find_source():
    for d in sorted(glob.glob('/kaggle/input/*')):
        if os.path.isdir(os.path.join(d, 'rufin')) or glob.glob(os.path.join(d, 'acts.jsonl*')):
            return d, None
    for z in sorted(glob.glob('/kaggle/input/*/*.zip')):
        return os.path.dirname(z), z
    return None, None

src, archive = find_source()
if src is None:
    print('входы, которые видит ноутбук:')
    for d in sorted(glob.glob('/kaggle/input/*')):
        print('   ', d, '->', sorted(os.listdir(d))[:6])
    raise SystemExit('датасет не подключён: панель справа -> + Add Input -> ru-fin-retrieval -> +')

print('источник:', src)
if archive:
    print('лежит архивом, распаковываю:', archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for item in glob.glob(os.path.join(src, '*')):
        dst = os.path.join('/kaggle/working', os.path.basename(item))
        if os.path.isdir(item):
            shutil.copytree(item, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dst)

os.chdir('/kaggle/working')
print('\nв рабочей папке:', sorted(os.listdir('.')))

assert os.path.isdir('rufin'), f'пакета rufin нет; в источнике было: {sorted(os.listdir(src))}'
# Kaggle иногда разжимает архивы, поэтому корпус может называться и так, и так
assert glob.glob('acts.jsonl*'), 'корпуса нет'
print('всё на месте:', sorted(glob.glob('acts.jsonl*')))


## Где лежат эмбеддинги этапа A

Путь зависит от того, как назван ноутбук этапа A, поэтому он ищется,
а не вписывается руками.


In [ ]:
import glob
found = glob.glob('/kaggle/input/*/embeddings')
print('найдено:', found)
EMB = found[0] if found else None
print('используем:', EMB)
assert EMB, 'вывод этапа A не подключён — добавьте его через + Add Input'


## Прогон

Около полутора часов; больше всего времени уходит на переранжирование
(двести запросов по пятьдесят кандидатов).


In [ ]:
!python run_phase_b.py \
    --queries /kaggle/input/ru-fin-queries/queries.jsonl \
    --embeddings {EMB} \
    --out /kaggle/working/runs


## Что получилось


In [ ]:
import json, os
r = json.load(open('/kaggle/working/runs/report_phase_b.json'))
print('запросов:', r['queries'])
for x in r['runs']:
    print(f"   {x['config']:<12} {x['retrieval']}")
size = sum(os.path.getsize(os.path.join('/kaggle/working/runs', f))
           for f in os.listdir('/kaggle/working/runs'))
print(f'\nвсего {size/1048576:.1f} МБ — скачать папку runs целиком')


Скачать папку `runs` и положить в проект как `data/runs`.
Дальше на маке: `make metrics`, `make errors`, `make latency`.
